# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsiddan25/program/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

A content review queue is generated by the model, which uses Feb through April signals to rank pages associated with a May impression decline. May performance is not used to assign reason codes or actions, and June (final month) is not used as May is treated as the test month. The score represents represents relative model-estimated decline risk. It is not a guarantee that a page will definitely decline or proof that the content needs refreshing.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)



Paste your Hugging Face READ token (hf_...): ··········


In [2]:
import os
import sys
import duckdb
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.13.15
pandas: 2.2.3
NumPy: 2.1.3
scikit-learn: 1.6.1


In [3]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": (
        f"read_parquet('{REL}/dim_content.parquet')"
    ),
    "fact_daily": (
        f"read_parquet("
        f"'{REL}/fact_content_daily_performance/**/*.parquet'"
        f")"
    ),
}

print("DuckDB connection and table paths are ready.")

DuckDB connection and table paths are ready.


In [4]:
MONTHS = [
    "2026-02",
    "2026-03",
    "2026-04",
    "2026-05",
    "2026-06",
]

monthly_queries = []

for month in MONTHS:
    monthly_queries.append(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            '{month}' AS month_key,

            COUNT(*) AS observed_days,

            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_sum_position) AS sum_position,

            AVG(gsc_impressions) AS daily_impression_mean,
            STDDEV_SAMP(gsc_impressions) AS daily_impression_std,

            STDDEV_SAMP(
                CASE
                    WHEN gsc_impressions > 0
                    THEN gsc_avg_position
                END
            ) AS daily_position_std,

            SUM(
                CASE
                    WHEN gsc_impressions > 0 THEN 1
                    ELSE 0
                END
            ) AS days_with_impressions

        FROM read_parquet(
            '{REL}/fact_content_daily_performance/'
            'month={month}/*.parquet'
        )

        WHERE gsc_data_available IS TRUE

        GROUP BY
            client_hash_id,
            content_hash_id
    """)

monthly_union_sql = "\nUNION ALL\n".join(monthly_queries)

monthly_summary = con.sql(monthly_union_sql).df()

monthly_summary = monthly_summary.sort_values(
    ["month_key", "client_hash_id", "content_hash_id"]
).reset_index(drop=True)

print(f"Monthly summary rows: {len(monthly_summary):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Monthly summary rows: 971,603


In [5]:
monthly_coverage = (
    monthly_summary
    .groupby("month_key")
    .agg(
        page_rows=("content_hash_id", "size"),
        clients=("client_hash_id", "nunique"),
        minimum_days=("observed_days", "min"),
        maximum_days=("observed_days", "max"),
    )
    .reset_index()
)

monthly_coverage

,month_key,page_rows,clients,minimum_days,maximum_days
0,2026-02,153559,46,1,28
1,2026-03,176738,47,1,31
2,2026-04,194760,51,1,30
3,2026-05,237910,56,1,31
4,2026-06,208636,55,1,30


In [6]:
monthly_cache_path = "/content/flyrank_monthly_summary.pkl"

monthly_summary.to_pickle(monthly_cache_path)

print(f"Temporary cache saved: {monthly_cache_path}")

Temporary cache saved: /content/flyrank_monthly_summary.pkl


In [7]:
monthly_value_columns = [
    "observed_days",
    "impressions",
    "clicks",
    "sum_position",
    "daily_impression_mean",
    "daily_impression_std",
    "daily_position_std",
    "days_with_impressions",
]


def select_month(month, prefix):
    month_frame = monthly_summary.loc[
        monthly_summary["month_key"] == month,
        [
            "client_hash_id",
            "content_hash_id",
            *monthly_value_columns,
        ],
    ].copy()

    rename_map = {
        column: f"{prefix}_{column}"
        for column in monthly_value_columns
    }

    return month_frame.rename(columns=rename_map)


february = select_month("2026-02", "feb")
march = select_month("2026-03", "mar")
april = select_month("2026-04", "apr")
may = select_month("2026-05", "may")

In [8]:
development = (
    february
    .merge(
        march,
        on=["client_hash_id", "content_hash_id"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        april,
        on=["client_hash_id", "content_hash_id"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        may,
        on=["client_hash_id", "content_hash_id"],
        how="inner",
        validate="one_to_one",
    )
)

development = development[
    (development["feb_observed_days"] >= 20)
    & (development["mar_observed_days"] >= 20)
    & (development["apr_observed_days"] >= 20)
    & (development["may_observed_days"] >= 20)
    & (development["mar_impressions"] > 0)
    & (development["apr_impressions"] > 0)
].copy()

development = development.reset_index(drop=True)

print(f"Development rows: {len(development):,}")
print(
    "Development clients:",
    development["client_hash_id"].nunique(),
)
print(
    "Duplicate client-page rows:",
    development.duplicated(
        ["client_hash_id", "content_hash_id"]
    ).sum(),
)

Development rows: 62,558
Development clients: 23
Duplicate client-page rows: 0


In [9]:
# Three-month traffic totals: February through April
development["impressions_90d"] = (
    development["feb_impressions"]
    + development["mar_impressions"]
    + development["apr_impressions"]
)

development["clicks_90d"] = (
    development["feb_clicks"]
    + development["mar_clicks"]
    + development["apr_clicks"]
)

# Log versions reduce the effect of extremely large traffic values.
development["log_impressions_90d"] = np.log1p(
    development["impressions_90d"]
)

development["log_previous_month_impressions"] = np.log1p(
    development["mar_impressions"]
)

development["log_current_month_impressions"] = np.log1p(
    development["apr_impressions"]
)

# March-to-April impression momentum
development["recent_change_pct"] = (
    100
    * (
        development["apr_impressions"]
        - development["mar_impressions"]
    )
    / development["mar_impressions"]
)

# Limit extreme growth percentages caused by small denominators.
development["recent_change_pct_clipped"] = (
    development["recent_change_pct"]
    .clip(lower=-100, upper=500)
)

# Monthly CTR
development["previous_ctr"] = (
    100
    * development["mar_clicks"]
    / development["mar_impressions"]
)

development["current_ctr"] = (
    100
    * development["apr_clicks"]
    / development["apr_impressions"]
)

development["ctr_change"] = (
    development["current_ctr"]
    - development["previous_ctr"]
)

# Impression-weighted average position
development["previous_avg_position"] = (
    development["mar_sum_position"]
    / development["mar_impressions"]
)

development["current_avg_position"] = (
    development["apr_sum_position"]
    / development["apr_impressions"]
)

# Positive means the page's average position became worse.
development["position_change"] = (
    development["current_avg_position"]
    - development["previous_avg_position"]
)

# Current-month traffic volatility
development["current_impression_cv"] = (
    development["apr_daily_impression_std"]
    / development["apr_daily_impression_mean"]
)

development["current_position_std"] = (
    development["apr_daily_position_std"]
)

development["current_impression_day_rate"] = (
    development["apr_days_with_impressions"]
    / development["apr_observed_days"]
)

# May outcome: validation only, never a feature
development["future_change_pct"] = (
    100
    * (
        development["may_impressions"]
        - development["apr_impressions"]
    )
    / development["apr_impressions"]
)

development["future_decline"] = (
    development["may_impressions"]
    < 0.80 * development["apr_impressions"]
).astype(int)

In [10]:
print(
    "Future-decline base rate:",
    f"{development['future_decline'].mean():.2%}",
)

print(
    "Infinite feature values:",
    np.isinf(
        development.select_dtypes(include="number")
    ).sum().sum(),
)

print(
    "Missing current position volatility:",
    development["current_position_std"].isna().sum(),
)

Future-decline base rate: 48.79%
Infinite feature values: 0
Missing current position volatility: 0


In [11]:
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
)

In [12]:
# ============================================================
# SECTION 2: RANDOM VERSUS CLIENT-GROUPED VALIDATION
# ============================================================

feature_columns = [
    # Traffic level
    "log_impressions_90d",
    "log_previous_month_impressions",
    "log_current_month_impressions",

    # Recent traffic direction
    "recent_change_pct_clipped",

    # CTR level and movement
    "previous_ctr",
    "current_ctr",
    "ctr_change",

    # Search position level and movement
    "previous_avg_position",
    "current_avg_position",
    "position_change",

    # Current-month stability
    "current_impression_cv",
    "current_position_std",
    "current_impression_day_rate",
]



X = (
    development[feature_columns]
    .reset_index(drop=True)
    .copy()
)

y = (
    development["future_decline"]
    .astype(int)
    .reset_index(drop=True)
)

groups = (
    development["client_hash_id"]
    .astype(str)
    .reset_index(drop=True)
)

print(f"Evaluation rows: {len(X):,}")
print(f"Clients: {groups.nunique()}")
print(f"Future-decline base rate: {y.mean():.2%}")
print(f"Features: {len(feature_columns)}")

Evaluation rows: 62,558
Clients: 23
Future-decline base rate: 48.79%
Features: 13


In [13]:
def build_random_forest():
    return Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=200,
                    max_depth=10,
                    min_samples_leaf=25,
                    class_weight="balanced_subsample",
                    n_jobs=-1,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

In [14]:
def generate_oof_predictions(
    splitter,
    X,
    y,
    groups,
    design_name,
    use_groups,
):
    # Every page will eventually receive one validation prediction.
    oof_scores = np.full(len(X), np.nan)

    fold_records = []

    if use_groups:
        split_iterator = splitter.split(
            X,
            y,
            groups=groups,
        )
    else:
        split_iterator = splitter.split(X, y)

    for fold_number, (train_index, validation_index) in enumerate(
        split_iterator,
        start=1,
    ):
        fold_model = build_random_forest()

        X_fold_train = X.iloc[train_index]
        y_fold_train = y.iloc[train_index]

        X_fold_validation = X.iloc[validation_index]
        y_fold_validation = y.iloc[validation_index]

        fold_model.fit(
            X_fold_train,
            y_fold_train,
        )

        oof_scores[validation_index] = (
            fold_model.predict_proba(
                X_fold_validation
            )[:, 1]
        )

        train_clients = set(groups.iloc[train_index])
        validation_clients = set(groups.iloc[validation_index])

        fold_records.append({
            "validation_design": design_name,
            "fold": fold_number,
            "training_rows": len(train_index),
            "validation_rows": len(validation_index),
            "training_clients": len(train_clients),
            "validation_clients": len(validation_clients),
            "overlapping_clients": len(
                train_clients.intersection(validation_clients)
            ),
            "validation_base_rate": (
                y_fold_validation.mean()
            ),
        })

    # Every row should have exactly one out-of-fold score.
    assert not np.isnan(oof_scores).any()

    return oof_scores, pd.DataFrame(fold_records)

In [15]:
grouped_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

grouped_oof_scores, grouped_fold_summary = (
    generate_oof_predictions(
        splitter=grouped_splitter,
        X=X,
        y=y,
        groups=groups,
        design_name="Client-grouped CV — after",
        use_groups=True,
    )
)

In [16]:
required_objects = [
    "development",
    "feature_columns",
    "X",
    "y",
    "groups",
    "grouped_oof_scores",
    "grouped_fold_summary",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing required objects: {missing_objects}"
)

assert len(development) == len(grouped_oof_scores)
assert grouped_fold_summary["overlapping_clients"].sum() == 0

print("Validated Week-6 scores are ready.")
print(f"Pages available: {len(development):,}")

Validated Week-6 scores are ready.
Pages available: 62,558


In [17]:
# Only identifiers and pre-May information enter the action queue.
queue_source_columns = [
    "client_hash_id",
    "content_hash_id",
    "mar_impressions",
    "apr_impressions",
    "recent_change_pct",
    "previous_ctr",
    "current_ctr",
    "ctr_change",
    "previous_avg_position",
    "current_avg_position",
    "position_change",
    "current_impression_cv",
    "current_position_std",
    "current_impression_day_rate",
]

action_queue = (
    development[queue_source_columns]
    .reset_index(drop=True)
    .copy()
)

action_queue["model_risk_score"] = np.asarray(
    grouped_oof_scores
)

assert len(action_queue) == len(grouped_oof_scores)
assert action_queue["model_risk_score"].notna().all()

print(f"Queue candidates: {len(action_queue):,}")

Queue candidates: 62,558


In [18]:
impression_cv_cutoff = (
    action_queue["current_impression_cv"]
    .quantile(0.75)
)

position_std_cutoff = (
    action_queue["current_position_std"]
    .quantile(0.75)
)

position_change_cutoff = (
    action_queue["position_change"]
    .quantile(0.75)
)

high_visibility_cutoff = (
    action_queue["apr_impressions"]
    .quantile(0.75)
)

reason_thresholds = pd.DataFrame({
    "threshold": [
        "High impression volatility",
        "High position volatility",
        "Large position worsening",
        "High April visibility",
    ],
    "value": [
        impression_cv_cutoff,
        position_std_cutoff,
        position_change_cutoff,
        high_visibility_cutoff,
    ],
})

display(reason_thresholds)

,threshold,value
0,High impression volatility,0.647244
1,High position volatility,12.142380
2,Large position worsening,3.653012
3,High April visibility,2681.750000


In [19]:
reason_conditions = [
    # Clear recent loss
    action_queue["recent_change_pct"] <= -20,

    # Highly unstable impression pattern
    (
        action_queue["current_impression_cv"]
        >= impression_cv_cutoff
    ),

    # Google position became worse
    (
        (action_queue["position_change"] > 0)
        & (
            action_queue["position_change"]
            >= position_change_cutoff
        )
    ),

    # Highly unstable daily Google position
    (
        action_queue["current_position_std"]
        >= position_std_cutoff
    ),

    # Valuable existing visibility
    (
        action_queue["apr_impressions"]
        >= high_visibility_cutoff
    ),
]

reason_codes = [
    "recent_impression_decline",
    "high_impression_volatility",
    "worsening_search_position",
    "high_position_volatility",
    "high_visibility_at_risk",
]

action_queue["reason_code"] = np.select(
    reason_conditions,
    reason_codes,
    default="elevated_model_risk",
)

In [20]:
action_mapping = {
    "recent_impression_decline": "review_for_refresh",
    "high_impression_volatility": "diagnose_traffic_instability",
    "worsening_search_position": "review_serp_and_relevance",
    "high_position_volatility": "diagnose_ranking_instability",
    "high_visibility_at_risk": "prioritize_manual_review",
    "elevated_model_risk": "manual_investigation",
}

action_queue["action_label"] = (
    action_queue["reason_code"]
    .map(action_mapping)
)

In [21]:
def create_reason_note(row):
    reason = row["reason_code"]

    if reason == "recent_impression_decline":
        return (
            f"April impressions were "
            f"{abs(row['recent_change_pct']):.1f}% "
            f"lower than March."
        )

    if reason == "high_impression_volatility":
        return (
            "April daily impression volatility "
            "was in the highest population quartile."
        )

    if reason == "worsening_search_position":
        return (
            f"Average Google position worsened by "
            f"{row['position_change']:.1f} positions."
        )

    if reason == "high_position_volatility":
        return (
            "April daily search-position volatility "
            "was in the highest population quartile."
        )

    if reason == "high_visibility_at_risk":
        return (
            "April impressions were in the highest "
            "population quartile."
        )

    return (
        "The grouped model assigned elevated relative "
        "risk without one dominant rule-based signal."
    )


action_queue["reason_note"] = action_queue.apply(
    create_reason_note,
    axis=1,
)

In [22]:
# If the defined decline occurs, the page loses at least
# approximately 20% of its April impressions.
action_queue["decline_threshold_impressions"] = (
    0.20 * action_queue["apr_impressions"]
).round(0)

# Rank primarily by the validated model score.
# April impressions break exact score ties.
action_queue = action_queue.sort_values(
    [
        "model_risk_score",
        "apr_impressions",
    ],
    ascending=[False, False],
).reset_index(drop=True)

action_queue["global_rank"] = (
    np.arange(1, len(action_queue) + 1)
)

# Also show priority within each client.
action_queue["client_rank"] = (
    action_queue
    .groupby("client_hash_id")
    .cumcount()
    + 1
)

In [25]:
queue_display_columns = [
    "global_rank",
    "client_hash_id",
    "content_hash_id",
    "model_risk_score",
    "action_label",
    "reason_code",
    "reason_note",
    "mar_impressions",
    "apr_impressions",
    "recent_change_pct",
    "current_avg_position",
    "position_change",
    "current_impression_cv",
    "current_position_std",
    "decline_threshold_impressions",
    "client_rank",
]


ranked_queue = (
    action_queue[queue_display_columns]
    .copy()
)

REVIEW_BATCH_SIZE = 20

top_review_batch = (
    ranked_queue
    .head(REVIEW_BATCH_SIZE)
    .copy()
)

print(f"Complete ranked queue: {len(ranked_queue):,} pages")
print(f"Immediate human-review batch: {len(top_review_batch)} pages")

display(top_review_batch)



# Confirm future information was not exported.
forbidden_export_columns = {
    "may_impressions",
    "future_change_pct",
    "future_decline",
}

assert not forbidden_export_columns.intersection(
    ranked_queue.columns
)

print("PASS: No outcome-window columns appear in the queue.")


Complete ranked queue: 62,558 pages
Immediate human-review batch: 20 pages


,global_rank,client_hash_id,content_hash_id,model_risk_score,action_label,reason_code,reason_note,mar_impressions,apr_impressions,recent_change_pct,current_avg_position,position_change,current_impression_cv,current_position_std,decline_threshold_impressions,client_rank
0,1,client_73cda7b4e4f265ea,content_2b509b842b6a4bfa,0.973374,review_for_refresh,recent_impression_decline,April impressions were 47.7% lower than March.,2520.0,1319.0,-47.658730,1.796816,-0.248026,0.709367,5.578418,264.0,1
1,2,client_73cda7b4e4f265ea,content_7e803a2c9663bf84,0.973020,review_for_refresh,recent_impression_decline,April impressions were 50.7% lower than March.,2006.0,988.0,-50.747757,1.628543,0.051274,0.781962,13.934287,198.0,2
2,3,client_73cda7b4e4f265ea,content_a9545a357cd2e358,0.972297,review_for_refresh,recent_impression_decline,April impressions were 41.1% lower than March.,2308.0,1360.0,-41.074523,1.977206,0.183445,0.797477,4.981421,272.0,3
3,4,client_73cda7b4e4f265ea,content_7db2ded21184014d,0.971355,review_for_refresh,recent_impression_decline,April impressions were 61.2% lower than March.,1746.0,678.0,-61.168385,1.240413,0.666530,0.705929,6.428087,136.0,4
4,5,client_73cda7b4e4f265ea,content_888780a6f4240d2f,0.971314,review_for_refresh,recent_impression_decline,April impressions were 39.2% lower than March.,17454.0,10604.0,-39.246018,1.259053,-0.326314,0.868648,6.672118,2121.0,5
5,6,client_73cda7b4e4f265ea,content_787b5229dfb9f4f6,0.971098,review_for_refresh,recent_impression_decline,April impressions were 53.7% lower than March.,3537.0,1637.0,-53.717840,1.849725,0.443731,0.904501,4.676572,327.0,6
6,7,client_23a62021009f63c4,content_f440467402fe9741,0.970342,review_for_refresh,recent_impression_decline,April impressions were 41.7% lower than March.,3461.0,2017.0,-41.722046,2.289539,-0.144440,1.176143,10.035358,403.0,1
7,8,client_73cda7b4e4f265ea,content_5e320f3ef275fe45,0.970318,review_for_refresh,recent_impression_decline,April impressions were 57.3% lower than March.,1519.0,649.0,-57.274523,1.953775,0.507429,1.049664,5.669579,130.0,7
8,9,client_fef1a8f436438636,content_44e965b61a7745b3,0.969537,review_for_refresh,recent_impression_decline,April impressions were 42.6% lower than March.,1593.0,915.0,-42.561205,2.582514,1.111076,0.769391,10.677582,183.0,1
9,10,client_3197e6291363b4db,content_542589b54e60a053,0.969278,diagnose_traffic_instability,high_impression_volatility,April daily impression volatility was in the h...,3347.0,4058.0,21.242904,1.694431,-1.351879,1.236945,18.260905,812.0,1


PASS: No outcome-window columns appear in the queue.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [27]:
ranked_queue["queue_status"] = "not_currently_selected"

ranked_queue.loc[
    ranked_queue["global_rank"] <= 20,
    "queue_status"
] = "immediate_review"

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

export_columns = [
    "global_rank",
    "client_rank",
    "client_hash_id",
    "content_hash_id",
    "model_risk_score",
    "queue_status",
    "action_label",
    "reason_code",
    "reason_note",
    "mar_impressions",
    "apr_impressions",
    "recent_change_pct",
    "current_avg_position",
    "position_change",
]

queue_export = ranked_queue[export_columns].copy()

queue_path = output_dir / "content_action_playbook_queue.csv"
queue_export.to_csv(queue_path, index=False)

print(f"Exported {len(queue_export):,} rows")
print(f"Saved to: {queue_path}")

Exported 62,558 rows
Saved to: work/outputs/content_action_playbook_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.